# Sesión 05 — Medallion Architecture: Capa Silver
## Pipeline Bronze → Silver con calidad de datos, deduplicación y enriquecimiento

**Dominio:** Retail — ventas de artículos deportivos  
**Runtime mínimo:** Databricks 13.3 LTS  
**Catálogo / Schema:** `dbassociate` / `default`  
**Volume fuente:** `/Volumes/dbassociate/default/vol_landing/session05/`

---
### ¿Qué construimos hoy?

Partimos de una tabla Bronze con datos crudos que contienen tres tipos de problemas deliberados:
- Un registro con `monto_total` negativo
- Un registro con `producto_id` nulo
- Un registro duplicado (misma orden, dos archivos distintos)
- Un producto que no existe en el catálogo (`PROD-104`)

Al final del laboratorio tendremos:
- `silver_ventas` — registros válidos, deduplicados y enriquecidos con datos del catálogo
- `silver_ventas_cuarentena` — registros rechazados con motivo documentado
- `silver_quality_log` — métrica de calidad del batch

---
### Estructura del notebook
1. Setup y carga de archivos
2. Diagnóstico: ¿qué problemas tiene el Bronze?
3. Validación y bifurcación a cuarentena
4. Deduplicación por llave natural
5. Generación de surrogate key (SHA-256)
6. Enriquecimiento con dimensión de productos
7. Escritura de tabla Silver
8. MERGE INTO: carga incremental
9. Delta Constraints como guardia secundaria
10. Log de métricas de calidad

---
## 1. Setup y carga de archivos

Definimos las variables de entorno y verificamos que los archivos CSV estén disponibles en el Volume.  
Todos los imports del laboratorio se concentran aquí para evitar errores de nombre en celdas posteriores.

In [0]:
# Importaciones — todas centralizadas para que el notebook funcione celda a celda
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import (
    col, count, when, lit, trim, lower,
    sha2, concat_ws, to_timestamp, to_date,
    row_number, broadcast, current_timestamp, current_date, coalesce
)
from pyspark.sql.window import Window

CATALOG     = "dbassociate"
SCHEMA      = "default"
LAB_PATH    = f"/Volumes/{CATALOG}/{SCHEMA}/vol_landing/session05"

# dbutils.fs.mkdirs(LAB_PATH)
print(f"Volume listo: {LAB_PATH}")

Sube los dos archivos CSV a la ruta anterior desde la UI de Databricks (**Catalog → Volumes → vol_landing → session05 → Upload**), luego ejecuta la celda siguiente para verificar.

In [0]:
# Verificar que los archivos están disponibles antes de continuar
archivos = dbutils.fs.ls(LAB_PATH)
nombres  = [a.name for a in archivos]

for n in nombres:
    print(" -", n)

assert "bronze_ventas.csv" in nombres,  "Falta bronze_ventas.csv"
assert "dim_productos.csv" in nombres,   "Falta dim_productos.csv"
print("Archivos verificados correctamente.")

In [0]:
# Schema explícito para bronze_ventas
# fecha_venta se lee como String porque en Bronze preservamos el valor original.
# La conversión a DateType ocurre en la capa Silver, con validación explícita.

schema_ventas = StructType([
    StructField("orden_id",            StringType(),  True),
    StructField("linea_id",            IntegerType(), True),
    StructField("producto_id",         StringType(),  True),
    StructField("cliente_id",          StringType(),  True),
    StructField("cantidad",            IntegerType(), True),
    StructField("precio_unitario",     DoubleType(),  True),
    StructField("monto_total",         DoubleType(),  True),
    StructField("fecha_venta",         StringType(),  True),
    StructField("canal",               StringType(),  True),
    StructField("estado",              StringType(),  True),
    StructField("ingestion_timestamp", StringType(),  True),
    StructField("source_file",         StringType(),  True),
])

bronze_ventas = (
    spark.read
    .option("header", "true")
    .schema(schema_ventas)
    .csv(f"{LAB_PATH}/bronze_ventas.csv")
)
bronze_ventas.createOrReplaceTempView("bronze_ventas")
print(f"Registros en Bronze: {bronze_ventas.count()}")
display(bronze_ventas)

In [0]:
# Catálogo de productos — schema inferido es aceptable porque dim_productos es controlada
dim_productos = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{LAB_PATH}/dim_productos.csv")
)
dim_productos.createOrReplaceTempView("dim_productos")
print(f"Productos en catálogo: {dim_productos.count()}")
display(dim_productos)

---
## 2. Diagnóstico: ¿qué problemas tiene el Bronze?

Antes de transformar, inspeccionamos los datos para entender exactamente qué vamos a corregir.  
Este paso no modifica nada — solo observa. En un pipeline productivo correspondería a una fase de **data profiling**.

Problemas esperados en este dataset:
| Problema | Registro | Motivo de rechazo planificado |
|---|---|---|
| `monto_total` negativo | ORD-004 | `monto_no_positivo` |
| `producto_id` nulo | ORD-005 | `producto_id_nulo` |
| Duplicado por re-entrega | ORD-006 | Se resuelve en deduplicación, no en cuarentena |
| Producto fuera de catálogo | ORD-009 (PROD-104) | Pasa a Silver con `_join_status = unmatched` |

In [0]:
# Nulos por columna — identifica qué columnas tienen datos faltantes
from pyspark.sql.functions import count as cnt

print("Nulos por columna:")
display(
    bronze_ventas.select([
        cnt(when(col(c).isNull(), 1)).alias(c)
        for c in bronze_ventas.columns
    ])
)

In [0]:
# Duplicados por llave natural (orden_id, linea_id)
# Un par que aparece más de una vez indica re-entrega del mismo archivo o error de origen

dups = (
    bronze_ventas
    .groupBy("orden_id", "linea_id")
    .agg(cnt("*").alias("n_apariciones"))
    .filter(col("n_apariciones") > 1)
)
print(f"Pares (orden_id, linea_id) duplicados: {dups.count()}")
display(dups)

In [0]:
# Montos negativos y productos nulos — resumen rápido antes de la validación formal
print("Montos negativos:", bronze_ventas.filter(col("monto_total") <= 0).count())
print("Productos nulos: ", bronze_ventas.filter(col("producto_id").isNull()).count())

---
## 3. Validación y bifurcación a cuarentena

Aplicamos las reglas de calidad de negocio como una columna calculada `rejection_reason`.  
Si un registro viola alguna regla, se desvía a `silver_ventas_cuarentena` con el motivo documentado.  
**Los registros rechazados no se eliminan** — se preservan para auditoría y resolución posterior.

Reglas implementadas:
- `producto_id` no puede ser nulo
- `monto_total` debe ser mayor que cero
- `canal` debe pertenecer al dominio válido
- `estado` debe pertenecer al dominio válido

La conversión de `fecha_venta` a `DateType` se hace aquí usando `to_date`.  
El dataset solo tiene fechas en formato `yyyy-MM-dd`, por lo que `to_date` con formato explícito es seguro.

In [0]:
CANALES_VALIDOS = ["ecommerce", "tienda_fisica", "mayorista"]
ESTADOS_VALIDOS = ["completada", "pendiente", "cancelada"]

# Normalización de strings antes de validar
ventas_norm = (
    bronze_ventas
    .withColumn("canal",       trim(lower(col("canal"))))
    .withColumn("estado",      trim(lower(col("estado"))))
    .withColumn("producto_id", trim(col("producto_id")))
    .withColumn("fecha_venta_dt", to_date(col("fecha_venta"), "yyyy-MM-dd"))
)

# Asignar motivo de rechazo — primera regla que falla gana
ventas_validadas = ventas_norm.withColumn(
    "rejection_reason",
    when(col("producto_id").isNull(),               "producto_id_nulo")
    .when(col("monto_total") <= 0,                  "monto_no_positivo")
    .when(~col("canal").isin(CANALES_VALIDOS),       "canal_invalido")
    .when(~col("estado").isin(ESTADOS_VALIDOS),      "estado_invalido")
    .otherwise(None)
)

ventas_validas    = ventas_validadas.filter(col("rejection_reason").isNull())

ventas_cuarentena = (
    ventas_validadas
    .filter(col("rejection_reason").isNotNull())
    .withColumn("rejection_timestamp", current_timestamp())
    .withColumn("source_batch_id",     lit("session05_lab"))
)

total      = ventas_validadas.count()
validos    = ventas_validas.count()
rechazados = ventas_cuarentena.count()


print(f"Total: {total}  |  Válidos: {validos}  |  Rechazados: {rechazados}")
print(f"Tasa de rechazo: {rechazados / total * 100:.1f}%")

print("\nMotivos de rechazo:")
display(
    ventas_cuarentena
    .groupBy("rejection_reason")
    .count()
    .orderBy("rejection_reason")
)


---
## 4. Deduplicación por llave natural

ORD-006 aparece dos veces porque fue enviado en dos archivos diferentes.  
Usamos `ROW_NUMBER` sobre la ventana particionada por `(orden_id, linea_id)`,  
ordenada por `ingestion_timestamp` descendente: **el registro más reciente gana**.  

Esta regla de desempate debe estar documentada y acordada con el equipo de negocio —  
"más reciente" no siempre es la decisión correcta en todos los dominios.

In [0]:
# Ventana de deduplicación: una fila por (orden_id, linea_id)
# El timestamp de ingesta determina cuál versión conservamos

ventana_dedup = (
    Window
    .partitionBy("orden_id", "linea_id")
    .orderBy(
        to_timestamp(col("ingestion_timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'").desc()
    )
)

ventas_dedup = (
    ventas_validas
    .withColumn("_rn", row_number().over(ventana_dedup))
    .filter(col("_rn") == 1)
    .drop("_rn")
)



antes  = ventas_validas.count()
despues = ventas_dedup.count()
print(f"Antes del dedup: {antes}  |  Después: {despues}  |  Eliminados: {antes - despues}")

# Confirmar que ORD-006 quedó una sola vez
display(
    ventas_dedup
    .filter(col("orden_id") == "ORD-006")
    .select("orden_id", "linea_id", "ingestion_timestamp", "source_file")
)




---
## 5. Surrogate key con SHA-256

Generamos una clave subrogada determinística: los mismos valores de `(orden_id, linea_id)`  
siempre producen el mismo hash, independientemente del cluster o la configuración de particiones.  

**Por qué no `monotonically_increasing_id()`:**  
Ese función genera IDs que dependen del número de particiones del job.  
Si la tabla se reprocesa en un cluster con distinta configuración, los IDs cambian  
y cualquier join histórico basado en esa columna se rompe silenciosamente.

In [0]:
ventas_dedup.limit(3).display()

In [0]:
# SHA-256 sobre la concatenación de las columnas que forman la llave natural
# El separador '|' evita colisiones entre valores como ('AB', 'C') y ('A', 'BC')

ventas_con_sk = ventas_dedup.withColumn(
    "venta_sk",
    sha2(
        concat_ws("|", col("orden_id"), col("linea_id").cast("string")),
        256
    )
)

print("Surrogate keys generados (muestra):")
display(
    ventas_con_sk.select("orden_id", "linea_id", "venta_sk").limit(5)
)

# Verificar unicidad: ningún hash debe repetirse
total_sk    = ventas_con_sk.count()
distintos_sk = ventas_con_sk.select("venta_sk").distinct().count()
print(f"\nRegistros: {total_sk}  |  SKs únicos: {distintos_sk}")
assert total_sk == distintos_sk, "Hay surrogate keys duplicados — revisar llave natural"

---
## 6. Enriquecimiento con dimensión de productos

Hacemos un LEFT JOIN contra `dim_productos` para agregar nombre, categoría y marca.  
Usamos `broadcast()` porque la dimensión es pequeña — evita un shuffle innecesario.

Usamos LEFT JOIN (no INNER) para no descartar registros silenciosamente.  
Un registro sin match en el catálogo (como ORD-009 con PROD-104) pasa a Silver  
con `_join_status = 'unmatched'` para que el equipo de datos lo investigue.

In [0]:
ventas_con_sk.display()

In [0]:
# Proyección de la dimensión antes del join para evitar columnas ambiguas
dim_enrich = dim_productos.select(
    col("producto_id"),
    col("nombre_producto"),
    col("categoria"),
    col("marca"),
)


ventas_enriquecidas = (
    ventas_con_sk
    .join(broadcast(dim_enrich), on="producto_id", how="left")
    .withColumn(
        "_join_status",
        when(col("nombre_producto").isNull(), "unmatched").otherwise("matched")
    )
)

matched   = ventas_enriquecidas.filter(col("_join_status") == "matched").count()
unmatched = ventas_enriquecidas.filter(col("_join_status") == "unmatched").count()
print(f"Matched: {matched}  |  Unmatched: {unmatched}")

if unmatched > 0:
    print("\nRegistros sin match en catálogo (investiga el producto_id):")
    display(
        ventas_enriquecidas
        .filter(col("_join_status") == "unmatched")
        .select("orden_id", "producto_id", "_join_status")
    )



---
## 7. Escritura de la tabla Silver

Seleccionamos explícitamente las columnas que van a Silver — nunca `SELECT *`.  
Incluimos `silver_processed_at` como columna técnica de auditoría.

La tabla de cuarentena se escribe por separado con las columnas adicionales  
`rejection_reason`, `rejection_timestamp` y `source_batch_id`.

In [0]:
# Proyección final para silver_ventas — columnas explícitas
silver_df = (
    ventas_enriquecidas
    .select(
        col("venta_sk"),
        col("orden_id"),
        col("linea_id"),
        col("producto_id"),
        col("nombre_producto"),
        col("categoria"),
        col("marca"),
        col("cliente_id"),
        col("cantidad"),
        col("precio_unitario"),
        col("monto_total"),
        col("fecha_venta_dt").alias("fecha_venta"),
        col("canal"),
        col("estado"),
        col("_join_status"),
        col("source_file"),
        to_timestamp(
            col("ingestion_timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'"
        ).alias("ingestion_timestamp"),
    )
    .withColumn("silver_processed_at", current_timestamp())
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_ventas")
)

print(f"silver_ventas escrita: {spark.table(f'{CATALOG}.{SCHEMA}.silver_ventas').count()} registros")
display(spark.table(f"{CATALOG}.{SCHEMA}.silver_ventas"))

In [0]:
# Tabla de cuarentena: registros rechazados con metadatos de rechazo
(
    ventas_cuarentena.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_ventas_cuarentena")
)

print(f"silver_ventas_cuarentena: {spark.table(f'{CATALOG}.{SCHEMA}.silver_ventas_cuarentena').count()} registros")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.silver_ventas_cuarentena")
    .select("orden_id", "producto_id", "monto_total", "rejection_reason", "rejection_timestamp")
)

---
## 8. MERGE INTO: carga incremental

Simulamos un segundo batch donde:
- **ORD-003** cambia de estado `pendiente` → `completada`
- **ORD-010** es un registro completamente nuevo

`MERGE INTO` aplica ambas operaciones en una sola pasada atómica:  
actualiza los existentes y agrega los nuevos sin duplicar ni perder datos.  
Si se re-ejecuta con el mismo staging, el resultado es idéntico — es idempotente.

In [0]:
# Construimos el staging como SQL literal — evita problemas de schema matching
# con createDataFrame en entornos con tipos estrictos

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW staging_incremental AS
SELECT
  sha2(concat_ws('|', orden_id, CAST(linea_id AS STRING)), 256) AS venta_sk,
  orden_id, linea_id, producto_id, nombre_producto, categoria, marca,
  cliente_id, cantidad, precio_unitario, monto_total,
  fecha_venta, canal, estado, _join_status, source_file,
  ingestion_timestamp, current_timestamp() AS silver_processed_at
FROM (
  VALUES
    ('ORD-003', 1, 'PROD-103', 'Bicicleta Urbana City', 'Ciclismo', 'VeloCity',
     'CLI-003', 3, 200.00, 600.00, DATE'2024-01-12', 'ecommerce', 'completada',
     'matched', 'ventas_inc.csv', TIMESTAMP'2024-01-20 08:00:00'),
    ('ORD-010', 1, 'PROD-101', 'Zapatilla Running X2', 'Calzado', 'SportMax',
     'CLI-010', 1, 150.00, 150.00, DATE'2024-01-19', 'ecommerce', 'completada',
     'matched', 'ventas_inc.csv', TIMESTAMP'2024-01-20 09:00:00')
) AS t(orden_id, linea_id, producto_id, nombre_producto, categoria, marca,
       cliente_id, cantidad, precio_unitario, monto_total,
       fecha_venta, canal, estado, _join_status, source_file, ingestion_timestamp)
""")

print("Staging incremental:")
display(spark.sql("SELECT * FROM staging_incremental"))

In [0]:
# MERGE INTO: match por venta_sk (surrogate key determinístico)
# WHEN MATCHED: actualiza solo si el estado cambió
# WHEN NOT MATCHED: inserta el registro nuevo

spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.silver_ventas AS target
USING staging_incremental AS source
  ON target.venta_sk = source.venta_sk
WHEN MATCHED AND target.estado != source.estado THEN
  UPDATE SET
    target.estado              = source.estado,
    target.silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN
  INSERT *
""")

print(f"Registros tras MERGE: {spark.table(f'{CATALOG}.{SCHEMA}.silver_ventas').count()}")

# Verificar el resultado de ambos casos
display(
    spark.table(f"{CATALOG}.{SCHEMA}.silver_ventas")
    .filter(col("orden_id").isin(["ORD-003", "ORD-010"]))
    .select("orden_id", "estado", "silver_processed_at")
)

In [0]:
display(spark.table(f"{CATALOG}.{SCHEMA}.silver_ventas"))

---
## 9. Delta Constraints como guardia secundaria

Los constraints de Delta Lake son una **segunda línea de defensa**, no la principal.  
La validación real ocurrió en la sección 3. Los constraints protegen contra errores  
en código futuro que intente escribir directamente en la tabla sin pasar por el pipeline.

Si cualquier registro viola el constraint, **la operación de escritura entera falla**:  
no hay descarte parcial ni warning silencioso.

In [0]:
# Agregar constraints de integridad a la tabla Silver
try:
    spark.sql(f"""
      ALTER TABLE {CATALOG}.{SCHEMA}.silver_ventas
      ADD CONSTRAINT chk_monto_positivo CHECK (monto_total > 0)
    """)
except Exception as e:
    if "CONSTRAINT_ALREADY_EXISTS" in str(e):
        print("Constraint chk_monto_positivo ya existe.")
    else:
        raise

try:
    spark.sql(f"""
      ALTER TABLE {CATALOG}.{SCHEMA}.silver_ventas
      ADD CONSTRAINT chk_cantidad_positiva CHECK (cantidad > 0)
    """)
except Exception as e:
    if "CONSTRAINT_ALREADY_EXISTS" in str(e):
        print("Constraint chk_cantidad_positiva ya existe.")
    else:
        raise

print("Constraints aplicados. Verificando:")
props = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{SCHEMA}.silver_ventas").collect()[0]["properties"]
for k, v in props.items():
    if "constraint" in k.lower():
        print(f"  {k}: {v}")

In [0]:
%sql
DESCRIBE DETAIL dbassociate.default.silver_ventas

In [0]:
# Demostración: intentar insertar un monto negativo — debe fallar con excepción clara
try:
    spark.sql(f"""
      INSERT INTO {CATALOG}.{SCHEMA}.silver_ventas
        (venta_sk, orden_id, linea_id, producto_id, cliente_id, cantidad,
         precio_unitario, monto_total, fecha_venta, canal, estado,
         _join_status, source_file, ingestion_timestamp, silver_processed_at)
      VALUES
        ('sk-test', 'ORD-TEST', 1, 'PROD-101', 'CLI-000', 1,
         100.0, -50.0, DATE'2024-01-01', 'ecommerce', 'completada',
         'matched', 'test.csv', current_timestamp(), current_timestamp())
    """)
    print("ERROR: la inserción debería haber fallado.")
except Exception as e:
    print("Constraint funcionó — inserción rechazada correctamente.")
    # Mostramos solo la primera línea del error para claridad
    print(str(e).split("\n")[0])

---
## 10. Log de métricas de calidad

Registramos las métricas del batch en una tabla de auditoría.  
En producción, esta tabla se alimenta en cada ejecución del pipeline  
y permite monitorear la **tasa de rechazo** a lo largo del tiempo.

Si la tasa de rechazo supera un umbral (ej. 10%), el pipeline debería generar una alerta  
antes de que el problema llegue a Gold o a los dashboards de negocio.

In [0]:
import datetime
from pyspark.sql.types import LongType, DoubleType, TimestampType

duplicados_eliminados = antes - despues     # variables de sección 4
total_silver_final    = spark.table(f"{CATALOG}.{SCHEMA}.silver_ventas").count()
tasa_rechazo          = round(rechazados / total * 100, 2)

schema_log = StructType([
    StructField("batch_id",               StringType()),
    StructField("processed_at",           TimestampType()),
    StructField("registros_bronze",       LongType()),
    StructField("registros_validos",      LongType()),
    StructField("registros_rechazados",   LongType()),
    StructField("duplicados_eliminados",  LongType()),
    StructField("registros_silver_final", LongType()),
    StructField("tasa_rechazo_pct",       DoubleType()),
])

log_df = spark.createDataFrame(
    [(
        "session05_lab",
        datetime.datetime.now(),
        int(total),
        int(validos),
        int(rechazados),
        int(duplicados_eliminados),
        int(total_silver_final),
        float(tasa_rechazo),
    )],
    schema=schema_log,
)

(
    log_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_quality_log")
)

print("Log registrado:")
display(spark.table(f"{CATALOG}.{SCHEMA}.silver_quality_log"))

---
## Resumen

| Tabla | Registros | Descripción |
|---|---|---|
| `silver_ventas` | 8 | Válidos, deduplicados, enriquecidos (incluye 1 unmatched) |
| `silver_ventas_cuarentena` | 2 | Rechazados por monto negativo y producto nulo |
| `silver_quality_log` | 1 batch | Tasa de rechazo, duplicados eliminados |

**Conceptos aplicados en este laboratorio:**
- `to_date` con formato explícito para conversión segura cuando el formato es conocido
- Validación multi-regla con `rejection_reason` y bifurcación a cuarentena
- `ROW_NUMBER` con ventana sobre llave natural para deduplicación reproducible
- `sha2(concat_ws(...))` como surrogate key determinístico
- `broadcast` join con columna `_join_status` para auditoría de integridad referencial
- `MERGE INTO` construido desde SQL literal para evitar problemas de tipo con `createDataFrame`
- Delta Constraints como segunda línea de defensa post-validación

**Siguiente sesión:** Gold — agregaciones, KPIs y optimización para consumo analítico.